In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import re
from typing import List

import pdfplumber
import requests

In [15]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str = "cl100k_base") -> int:
    encoding = tiktoken.get_encoding(encoding_name)
    return len(encoding.encode(string))


In [36]:
import sys
sys.path.insert(0, "..")

from utils import chat

In [ ]:
stepback_system_message = """
You are an expert at world knowledge. Your task is to step back
and paraphrase a question to a more generic step-back question, which
is easier to answer. Here are a few examples

"input": "Could the members of The Police perform lawful arrests?"
"output": "what can the members of The Police do?"

"input": "Jan Sindel's was born in what country?"
"output": "what is Jan Sindel's personal history?"

ONLY RETURN THE STEP-BACK QUESTION, DO NOT RETURN ANY OTHER TEXT.
"""


def generate_stepback(question: str):
    step_back_question = chat(
        model="llama3.2",
        messages=[
            {"role": "system", "content": stepback_system_message},
            {"role": "user", "content": question},
        ],
    )
    return step_back_question


In [10]:
question = "Which team did Thierry Audel play for from 2007 to 2008?"
step_back_question = generate_stepback(question)
print(f"Stepback results: {step_back_question}")

Stepback results: "what teams has Thierry Audel played for?"


In [11]:
remote_pdf_url = "https://arxiv.org/pdf/1709.00666.pdf"
pdf_filename = "../data/ch03-downloaded.pdf"

response = requests.get(remote_pdf_url)

if response.status_code == 200:
    with open(pdf_filename, "wb") as pdf_file:
        pdf_file.write(response.content)
else:
    print("Failed to download the PDF. Status code:", response.status_code)

In [12]:
text = ""

with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

In [13]:
def split_text_by_titles(text):
    # A regular expression pattern for titles that
    # match lines starting with one or more digits, an optional uppercase letter,
    # followed by a dot, a space, and then up to 50 characters
    title_pattern = re.compile(r"(\n\d+[A-Z]?\. {1,3}.{0,60}\n)", re.DOTALL)
    titles = title_pattern.findall(text)
    # Split the text at these titles
    sections = re.split(title_pattern, text)
    sections_with_titles = []
    # Append the first section
    sections_with_titles.append(sections[0])
    # Iterate over the rest of sections
    for i in range(1, len(titles) + 1):
        section_text = sections[i * 2 - 1].strip() + "\n" + sections[i * 2].strip()
        sections_with_titles.append(section_text)

    return sections_with_titles


sections = split_text_by_titles(text)
print(f"Number of sections: {len(sections)}")

Number of sections: 9


In [14]:
# Testing langchain's RecursiveCharacterTextSplitter with the same regex pattern
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=[r"\n\d+[A-Z]?\. {1,3}.{0,60}\n"],
    is_separator_regex=True,
    keep_separator=True,
)
sections_lang = splitter.split_text(text)
print(f"Number of sections (langchain): {len(sections_lang)}")

/Users/ialvarengadesousa/Projects/studies/graphrag_neo4j/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of sections (langchain): 7


In [22]:
print("Token counts per section:")
for s in sections:
    print(num_tokens_from_string(s))

Token counts per section:
154
254
4186
570
2703
804
637
194
600


In [18]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=40)

parent_chunks = []
for s in sections:
    parent_chunks.extend(text_splitter.split_text(s))

In [23]:
print("Token counts per parent chunk:")
for pc in parent_chunks:
    print(num_tokens_from_string(pc))

Token counts per parent chunk:
154
254
418
473
430
462
449
429
426
453
439
211
424
145
432
418
507
460
427
425
29
457
347
432
209
194
411
188


In [24]:
cypher_import_query = """
MERGE (pdf:PDF {id:$pdf_id})
MERGE (p:Parent {id:$pdf_id + '-' + $id})
SET p.text = $parent
MERGE (pdf)-[:HAS_PARENT]->(p)
WITH p, $children AS children, $embeddings as embeddings
UNWIND range(0, size(children) - 1) AS child_index
MERGE (c:Child {id: $pdf_id + '-' + $id + '-' + toString(child_index)})
SET c.text = children[child_index], c.embedding = embeddings[child_index]
MERGE (p)-[:HAS_CHILD]->(c);
"""

In [ ]:
from utils import embed


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2056.63it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from utils import init_neo4j_driver

neo4j_driver = init_neo4j_driver()


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)

for i, chunk in enumerate(parent_chunks):
    child_chunks = text_splitter.split_text(chunk)
    embeddings = embed(child_chunks)
    # Add to neo4j
    neo4j_driver.execute_query(
        cypher_import_query,
        id=str(i),
        pdf_id="1709.00666",
        parent=chunk,
        children=child_chunks,
        embeddings=embeddings,
    )


In [29]:
neo4j_driver.execute_query("""CREATE VECTOR INDEX parent IF NOT EXISTS
FOR (c:Child)
ON c.embedding""")

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x1679c3ba0>, keys=[])

In [30]:
retrieval_query = """
CALL db.index.vector.queryNodes($index_name, $k * 4, $question_embedding)
YIELD node, score
MATCH (node)<-[:HAS_CHILD]-(parent)
WITH parent, max(score) AS score
RETURN parent.text AS text, score
ORDER BY score DESC
LIMIT toInteger($k)
"""

In [ ]:
def parent_retrieval(question: str, k: int = 4) -> List[str]:
    question_embedding = embed(question)

    similar_records, _, _ = neo4j_driver.execute_query(
        retrieval_query,
        question_embedding=question_embedding,
        k=k,
        index_name="parent",
    )

    return [record["text"] for record in similar_records]


In [32]:
documents = parent_retrieval(
    "Who was the Einsten's collaborator on sound reproduction system?"
)
for d in documents:
    print(d)
    print("=" * 20)

113B. Sound reproduction system with Rudolf Goldschmidt
Rudolf Goldschmidt was a German Engineer and inventor. He earned his engineering degree
in 1898 and PhD in 1906. He spent a decade working in England with major firms such as Crampton,
Arc works, Westinghouse etc. On returning back to Germany he joined Darmstadt T H University as a
professor. Goldschmidt was a prolific inventor. His first patent was for a bicycle gear while still an
engineering student. In 1908 he developed a rotating radio‐frequency machine, which was used as an
early radio transmitter. The transmitter was used in the first trans‐Atlantic radiotelegraphic link
between Germany and United States, opened on 19th June, 1914, with an exchange of telegrams
between Kaiser Wilhelm II and President Woodrow Wilson.
Figure 5: Einstein‐Goldschmidt design of a sound reproduction system.
In 1922 Goldschmidt approached Einstein for his expert opinion regarding one of his patents.
Thereafter, they kept in touch. Even after Einst

In [35]:
generate_stepback("Who was the Einsten's collaborator on sound reproduction system?")

'What researcher worked closely with Einstein?'

In [37]:
answer_system_message = "You're en Einstein expert, but can only use the provided documents to respond to the questions."


def generate_answer(question: str, documents: List[str]) -> str:
    user_message = f"""
    Use the following documents to answer the question that will follow:
    {documents}

    ---

    The question to answer using information only from the above documents: {question}
    """
    result = chat(
        messages=[
            {"role": "system", "content": answer_system_message},
            {"role": "user", "content": user_message},
        ]
    )
    print("Response:", result)

In [38]:
def rag_pipeline(question: str) -> str:
    stepback_prompt = generate_stepback(question)
    print(f"Stepback prompt: {stepback_prompt}")
    documents = parent_retrieval(stepback_prompt)
    answer = generate_answer(question, documents)
    return answer

In [39]:
rag_pipeline("Who was the Einsten's collaborator on sound reproduction system?")

Stepback prompt: What was Einstein involved in?
Response: According to the document, Einstein's collaborator on the sound reproduction system was Rudolf Goldschmidt.


In [40]:
rag_pipeline("Who was the Einsten's first wife?")

Stepback prompt: what is Albert Einstein's personal history?
Response: There is no information provided in the provided documents about Einstein's first wife.


In [41]:
rag_pipeline("What age was Einstein when he died?")

Stepback prompt: What age do people typically live at?
Response: There is no information provided in the documents regarding the age of Einstein when he died. The documents only provide information about his birth, education, career, and personal life, but do not include the date and age of his death.


In [42]:
rag_pipeline("What are the main contributions of Einstein to physics?")

Stepback prompt: what does Einstein achieve in the field of physics?
Response: According to the document "63. Einstein's Inventions and Patents" and other sections, Einstein's main contributions to physics include:

1. Explaining the photoelectric effect using quantum mechanics, which was the first application of quantum mechanics in a physical process other than radiation.
2. Contributing to the development of relativity, although it was recognized by only a few scholars on its time, such as Max Planck, who gave a colloquium on relativity immediately after the paper was published.

It is worth noting that the document does not mention his groundbreaking work on the theory of general relativity, unification of gravity and electromagnetism, or other famous contributions of Einstein to physics. The information provided only briefly touches upon his explanation of the photoelectric effect and his recognition of contributions to relativity.
